<a href="https://colab.research.google.com/github/shaktidharreddy/Agentic_AI_eduhubspot_aug_sep_2026/blob/main/ModuleGAI_1/Foundations_of_Generative_AI_%26_Large_Language_Models_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Lab 1: Explore tokenization with tiktoken — analyze token counts, vocabulary sizes, and encoding strategies across GPT-4o and open-source tokenizers; visualize token boundaries on code vs natural language**

**Install required libraries**

In [ ]:
!pip -q install tiktoken transformers sentencepiece pandas matplotlib

**Login to Hugging Face**

In [ ]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


**Import libraries and load both tokenizers**

In [ ]:
import tiktoken
from transformers import AutoTokenizer
from IPython.display import display, HTML
import random

gpt_tokenizer = tiktoken.encoding_for_model("gpt-4o")

mistral_tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.3")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

**Compare vocabulary sizes**

In [ ]:
print("GPT-4o Vocabulary Size:", gpt_tokenizer.n_vocab)
print("Mistral Vocabulary Size:", len(mistral_tokenizer.vocab))

GPT-4o Vocabulary Size: 200019
Mistral Vocabulary Size: 32768


**Token count comparison**

In [ ]:
nl_text = "Hello, how are you doing today? I hope you are having a great day learning about tokenization!"
code_text = """def calculate_sum(a: int, b: int) -> int:
    return a + b
print(calculate_sum(10, 20))"""

print("=== Natural Language ===")
print("GPT-4o tokens:", len(gpt_tokenizer.encode(nl_text)))
print("Mistral tokens:", len(mistral_tokenizer.encode(nl_text)))

print("\n=== Code ===")
print("GPT-4o tokens:", len(gpt_tokenizer.encode(code_text)))
print("Mistral tokens:", len(mistral_tokenizer.encode(code_text)))

=== Natural Language ===
GPT-4o tokens: 21
Mistral tokens: 22

=== Code ===
GPT-4o tokens: 30
Mistral tokens: 38


**Visualize token boundaries**

In [ ]:
def visualize_tokens(text, tokenizer, name):
    if isinstance(tokenizer, tiktoken.Encoding):
        token_ids = tokenizer.encode(text)
        token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    else:
        token_ids = tokenizer.encode(text, add_special_tokens=False)
        token_strs = [tokenizer.decode([tid]) for tid in token_ids]

    colored = ""
    for token_str in token_strs:
        color = f"#{random.randint(0, 0xFFFFFF):06x}"
        colored += f'<span style="background-color:{color}; padding:2px; margin:1px; border-radius:3px;">{token_str}</span>'
    display(HTML(f"<h3>{name}</h3>{colored}"))
    print(f"Token count: {len(token_ids)}\n")

visualize_tokens(nl_text, gpt_tokenizer, "GPT-4o — Natural Language")
visualize_tokens(code_text, gpt_tokenizer, "GPT-4o — Code")
visualize_tokens(nl_text, mistral_tokenizer, "Mistral — Natural Language")
visualize_tokens(code_text, mistral_tokenizer, "Mistral — Code")

Token count: 21



Token count: 30



Token count: 21



Token count: 37



**Lab 2: Run inference on a pre-trained LLM (LLaMA 3 / Mistral 7B) via Hugging Face Transformers — compare outputs with varying temperature, top-p, top-k, and repetition penalty settings**

**Install Required Libraries**

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

print("Libraries installed!")

Libraries installed!


**Login to Hugging Face**

In [ ]:
from huggingface_hub import whoami, login

try:
    whoami()
    print("Already logged in!")
except:
    print("Login to Hugging Face:")
    login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Already logged in!


**Load Model & Tokenizer**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading model... (First time 4-8 minutes. Please wait - do not stop the cell)")

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Model loaded successfully!")

Loading model... (First time 4-8 minutes. Please wait - do not stop the cell)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully!


**Define the Generation Function**

In [ ]:
def generate_with_params(prompt, temp=0.7, top_p=0.9, top_k=50, rep_penalty=1.0, max_new=250):
    formatted_prompt = f"[INST] {prompt} [/INST]"

    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new,
        do_sample=True,
        temperature=temp,
        top_p=top_p,
        top_k=top_k,
        repetition_penalty=rep_penalty,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = full_text.split("[/INST]")[-1].strip()

    print(f"\n Settings: temp={temp} | top_p={top_p} | top_k={top_k} | rep_penalty={rep_penalty}")
    print("=" * 85)
    print(generated_text)
    print("=" * 85)

**Run Inference with Different Settings**

In [ ]:
prompt = "Explain quantum computing in simple terms suitable for a 12-year-old."

print("Generating responses with different settings...\n")

generate_with_params(prompt, temp=0.1, top_p=0.9, top_k=50, rep_penalty=1.0)

generate_with_params(prompt, temp=1.0, top_p=0.95, top_k=0, rep_penalty=1.0)

generate_with_params(prompt, temp=0.7, top_p=0.5, top_k=20, rep_penalty=1.0)

generate_with_params(prompt, temp=0.7, top_p=0.9, top_k=50, rep_penalty=1.8)

print("\n All generations completed successfully! ")

Generating responses with different settings...


 Settings: temp=0.1 | top_p=0.9 | top_k=50 | rep_penalty=1.0
Explain quantum computing in simple terms suitable for a 12-year-old.  Quantum computing is a type of computing that uses the principles of quantum mechanics, a branch of physics that describes the strange and mysterious behavior of particles at a very small scale.

In a regular computer, information is stored and processed using bits, which can be either a 0 or a 1. But in a quantum computer, we use something called a quantum bit, or qubit. A qubit can be both 0 and 1 at the same time, thanks to a property called superposition. This means that a quantum computer can process a lot more information than a regular computer at once.

Another important property of qubits is entanglement. When two qubits are entangled, they become linked in such a way that the state of one qubit affects the state of the other, no matter how far apart they are. This allows quantum computers to perfo